# Reasoning and Checking Model Answers

**Student:** Trenton Bailey  
**Course:** CSCI 1108  
**Date:** November 16, 2026

This notebook walks through deciding whether proposed answers from a language model should be accepted. It covers parsing answers from text, counting repeated answers with voting, and checking results against an independent calculation. The offline build uses scripted fixtures, not measured model behavior.

## Before You Code

A language model generates text using learned weights. Training changes those weights; inference uses them as-is. Editing a prompt changes the request, not the weights.

- **Zero-shot:** a prompt with no worked example
- **One-shot:** a prompt with one example question and answer
- **Self-consistency:** sample several solutions and aggregate final answers
- **Verifier:** checks requirements or evidence separately from the candidates

Repeated answers show agreement. An independent check can say something about correctness. These are different.

## Step 1 — Model the Inventory Changes

A shared workshop tool room starts with 18 drills. Students check out 5, another group returns 3. On paper: 18 - 5 + 3 = 16. The `expected` variable is our independently calculated reference, not an answer from a model.

In [1]:
problem = "18 drills in stock. 5 checked out, 3 returned. How many available?"
start = 18
checked_out = 5
returned = 3
expected = start - checked_out + returned
print(problem)
print("Reference count:", expected)

18 drills in stock. 5 checked out, 3 returned. How many available?
Reference count: 16


**Checkpoint:** Reference count is 16. Stopping at 13 would miss the return event entirely — the calculation must include both the checkout (subtraction) and the return (addition).

## Step 2 — Construct Three Requests

`direct` is zero-shot (one message, no example). `brief` is also zero-shot but asks for a short calculation. `worked` is one-shot: it has an example user message, an example assistant response, and the new task. Three messages but only one demonstration — the third message is the actual question, not a second example.

In [2]:
format_rule = "Use a final line in this format: Answer: <integer>"
direct = [{"role": "user", "content": problem + " " + format_rule}]
brief_text = problem + " Show a short calculation. " + format_rule
brief = [{"role": "user", "content": brief_text}]
worked = [
    {"role": "user",
     "content": "10 cords in stock. 4 borrowed, 2 returned. How many available?"},
    {"role": "assistant", "content": "10 - 4 = 6, then 6 + 2 = 8.\nAnswer: 8"},
    {"role": "user", "content": problem + " Show a short calculation. " + format_rule},
]
print("Direct message count:", len(direct))
print("Worked message count:", len(worked))
print(worked[1]["content"])

Direct message count: 1
Worked message count: 3
10 - 4 = 6, then 6 + 2 = 8.
Answer: 8


**Checkpoint:** Message counts are 1 and 3. The example answer ends with `Answer: 8`. Three messages still have only one demonstration because the third message is the new question for the model, not a second worked example.

## Step 3 — Supply Controlled Test Responses

These five fixtures are predetermined inputs, not model outputs. The first, second, and fourth omit the returned drills. The third includes both events. The fifth gives the right quantity in words but breaks the required format. Editing a prompt will not change these strings.

In [3]:
scripted = [
    "18 - 5 = 13.\nAnswer: 13",
    "Only the checkout matters.\nAnswer: 13",
    "18 - 5 + 3 = 16.\nAnswer: 16",
    "I counted thirteen available.\nAnswer: 13",
    "Sixteen drills are available.",
]
for text in scripted:
    print(repr(text))
print("Source: SCRIPTED TEST FIXTURES")

'18 - 5 = 13.\nAnswer: 13'
'Only the checkout matters.\nAnswer: 13'
'18 - 5 + 3 = 16.\nAnswer: 16'
'I counted thirteen available.\nAnswer: 13'
'Sixteen drills are available.'
Source: SCRIPTED TEST FIXTURES


**Checkpoint:** Five strings print. These are scripted fixtures — no model generates them.

## Step 4 — Convert Final Answer Text to an Integer

The parser checks the last line for the `Answer:` prefix. `int()` converts the text after the prefix to an integer. If the prefix is missing or the text is not a valid integer literal, the function returns `None`. `None` means no usable answer, which is different from the integer zero. We never use `eval` or `exec` on model responses.

In [4]:
def parse_answer(text):
    lines = text.strip().splitlines()
    if not lines:
        return None
    final_line = lines[-1].strip()
    prefix = "Answer:"
    if not final_line.startswith(prefix):
        return None
    number_text = final_line[len(prefix):].strip()
    try:
        return int(number_text)
    except ValueError:
        return None

answers = []
for text in scripted:
    answers.append(parse_answer(text))
print(answers)

[13, 13, 16, 13, None]


**Checkpoint:** `[13, 13, 16, 13, None]`. If we tried `Answer: 16 drills`, the `int()` call would fail on the word "drills" and return `None`. The parser accepts any integer format that Python `int()` handles, including a leading sign, but requires the `Answer:` label on the last nonblank line.

## Step 5 — Count the Votes

The voting function counts each usable integer. `None` values are skipped. If there is a unique most frequent value (a plurality), it is selected. A plurality does not always exceed half the valid votes. In this fixture, 13 has three of four valid votes and is still wrong.

In [5]:
def vote_answers(values):
    counts = {}
    for value in values:
        if value is None:
            continue
        if value not in counts:
            counts[value] = 0
        counts[value] = counts[value] + 1
    if not counts:
        return {"answer": None, "counts": counts, "status": "no valid answers"}
    highest_count = max(counts.values())
    leaders = []
    for value in counts:
        if counts[value] == highest_count:
            leaders.append(value)
    if len(leaders) > 1:
        return {"answer": None, "counts": counts, "status": "tie"}
    return {"answer": leaders[0], "counts": counts, "status": "selected"}

vote = vote_answers(answers)
print(vote)
print("Matches reference:", vote["answer"] == expected)

{'answer': 13, 'counts': {13: 3, 16: 1}, 'status': 'selected'}
Matches reference: False


**Checkpoint:** `counts` is `{13: 3, 16: 1}`, selected answer is 13, and `Matches reference` is `False`. The shared mistake is that three of the four fixtures forgot the return step and only computed 18 - 5 = 13. Voting picks the most popular answer, which is wrong here because the same error was repeated.

## Step 6 — Check the Final Outcome

The verifier compares each candidate's parsed answer against the independently calculated `expected` value. A match earns score 1; a wrong answer or format error earns 0. This is a rule-based outcome check, not a trained reward model.

In [6]:
def score_candidate(text, expected_answer):
    answer = parse_answer(text)
    if answer is None:
        status = "format error"
    elif answer == expected_answer:
        status = "correct"
    else:
        status = "wrong answer"
    score = 0
    if status == "correct":
        score = 1
    return {"text": text, "answer": answer, "score": score, "status": status}

records = []
for text in scripted:
    records.append(score_candidate(text, expected))
for record in records:
    print(record["answer"], record["score"], record["status"])

13 0 wrong answer
13 0 wrong answer
16 1 correct
13 0 wrong answer
None 0 format error


**Checkpoint:** Scores are 0, 0, 1, 0, 0. The last status is `format error`. The reference must come from somewhere other than agreement with these candidates, because the majority of them are wrong.

## Step 7 — Select by Score

Best-of-N means choosing from N candidates using a scoring rule. Here N is 5. The function returns the first candidate with score 1. If none pass, it returns `None`. This selector does not rank explanations, repair responses, or create a missing answer.

In [7]:
def select_best(records):
    for record in records:
        if record["score"] == 1:
            return record
    return None

best = select_best(records)
if best is None:
    print("No candidate passed.")
else:
    print("Selected count:", best["answer"])
    print("Original response:", best["text"])
print("Pool size:", len(records))

Selected count: 16
Original response: 18 - 5 + 3 = 16.
Answer: 16
Pool size: 5


**Checkpoint:** The verifier selects 16 from the same pool where voting selected 13. Pool size is 5. If the third fixture were removed, nothing would pass — the verifier would return `None`.

## Step 8 — Inspect the Displayed Intermediate States

A process check verifies the transitions, not just the final number. `[18, 13, 16]` has valid transitions: 18 - 5 = 13, then 13 + 3 = 16. `[18, 14, 16]` ends at the right count but both changes are wrong. A final-answer check sees only the last number; a process check can catch invalid transitions.

In [8]:
def check_process(states, start_count, out_count, return_count):
    if len(states) != 3:
        return [False, False, False]
    return [
        states[0] == start_count,
        states[1] == states[0] - out_count,
        states[2] == states[1] + return_count,
    ]

correct_steps = [18, 13, 16]
lucky_final = [18, 14, 16]
print("Valid states:", check_process(correct_steps, start, checked_out, returned))
print("Invalid transitions:", check_process(lucky_final, start, checked_out, returned))
print("Same final value:", correct_steps[-1] == lucky_final[-1] == expected)

Valid states: [True, True, True]
Invalid transitions: [True, False, False]
Same final value: True


**Checkpoint:** Valid states: `[True, True, True]`. Invalid transitions: `[True, False, False]`. Same final value: `True`. The second list fails because 18 - 5 is 13, not 14 — and then 14 + 3 is 17, not 16. The final 16 matches by coincidence; the transitions are both wrong.

## Optional Extension — Local Model with Ollama

Steps 9-11 use Ollama running locally to send real requests to `gemma3:1b`. The live switches control whether actual requests are sent.

### Step 9 — Define the Request Function

In [9]:
import json
import time
import urllib.request

MODEL = "gemma3:1b"
URL = "http://localhost:11434/v1/chat/completions"

def ask_model(messages, temperature=0.0):
    payload = {
        "model": MODEL, "messages": messages,
        "temperature": temperature, "max_tokens": 180, "stream": False,
    }
    request = urllib.request.Request(
        URL, data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    started = time.perf_counter()
    with urllib.request.urlopen(request, timeout=180) as reply:
        data = json.loads(reply.read().decode("utf-8"))
    choices = data.get("choices", [])
    if not choices:
        raise ValueError("The server returned no completion choices")
    return {
        "text": choices[0]["message"].get("content") or "",
        "model": data.get("model", MODEL),
        "usage": data.get("usage", {}),
        "seconds": time.perf_counter() - started,
    }

**Checkpoint:** No output expected. The line `urllib.request.urlopen(request, timeout=180)` sends the request. The path: Python dictionary → `json.dumps` → `.encode("utf-8")` → bytes sent over HTTP → response bytes → `.decode("utf-8")` → `json.loads` → Python dictionary.

### Step 10 — Compare Prompts

In [10]:
RUN_LIVE = False
live_comparisons = []
if RUN_LIVE:
    styles = [("direct", direct), ("one-shot", worked), ("brief calculation", brief)]
    for label, messages in styles:
        response = ask_model(messages, temperature=0.0)
        scored = score_candidate(response["text"], expected)
        comparison = {"style": label, "response": response, "scored": scored}
        live_comparisons.append(comparison)
        print(label, scored["status"], round(response["seconds"], 2))
        print(response["text"])
        print("Usage:", response["usage"])
else:
    print("Prompt comparison skipped; no request sent.")

Prompt comparison skipped; no request sent.


**Checkpoint:** With `False`, the skipped message prints. With `True`, results are model-dependent.

### Step 11 — Sample and Select

In [11]:
RUN_SAMPLING = False
N = 5
live_samples = []
if RUN_SAMPLING:
    for sample_number in range(N):
        response = ask_model(brief, temperature=0.8)
        scored = score_candidate(response["text"], expected)
        live_samples.append({"response": response, "scored": scored})
        print(sample_number + 1, scored["answer"], scored["status"])
    live_answers = []
    live_records = []
    for sample in live_samples:
        live_answers.append(sample["scored"]["answer"])
        live_records.append(sample["scored"])
    print("Vote:", vote_answers(live_answers))
    winner = select_best(live_records)
    if winner is None:
        print("No live candidate passed.")
    else:
        print("Verifier selection:", winner["answer"])
else:
    print("Sampling skipped; no request sent.")

Sampling skipped; no request sent.


**Checkpoint:** `False` skips all five requests.

## Practice and Review

Testing predictions for edge cases before running them.

In [12]:
# Predicted: tie (16 and 13 each have 1 vote)
print("vote_answers([16, 13]):", vote_answers([16, 13]))

# Predicted: no valid answers
print("vote_answers([None]):", vote_answers([None]))

# Predicted: selected 0 (zero is a valid integer, not None)
print("vote_answers([0, 0, None]):", vote_answers([0, 0, None]))

# Parser edge cases
print("parse empty:", parse_answer(""))
print("parse Answer: 16:", parse_answer("Answer: 16"))
print("parse Answer: 16 drills:", parse_answer("Answer: 16 drills"))
print("parse multiline:", parse_answer("Some reasoning.\nAnswer: 16"))

vote_answers([16, 13]): {'answer': None, 'counts': {16: 1, 13: 1}, 'status': 'tie'}
vote_answers([None]): {'answer': None, 'counts': {}, 'status': 'no valid answers'}
vote_answers([0, 0, None]): {'answer': 0, 'counts': {0: 2}, 'status': 'selected'}
parse empty: None
parse Answer: 16: 16
parse Answer: 16 drills: None
parse multiline: 16


The tie result is expected — both values have equal frequency. `None` inputs are skipped, so a list with only `None` has no valid answers. Zero is a valid integer and gets selected. `Answer: 16 drills` fails because `int("16 drills")` raises `ValueError`. These are formatting failures — the number itself is correct but the format does not match.

### Changed Problem: 24 Drills, 6 Out, 2 Returned

In [13]:
# New problem: 24 drills, 6 checked out, 2 returned
new_start = 24
new_out = 6
new_ret = 2
new_expected = new_start - new_out + new_ret
print("New reference count:", new_expected)

new_fixtures = [
    "24 - 6 + 2 = 20.\nAnswer: 20",   # correct
    "24 - 6 = 18.\nAnswer: 18",        # wrong, forgot return
    "24 - 4 = 20.\nAnswer: 20",        # correct number, wrong reasoning
]

new_answers = []
for text in new_fixtures:
    new_answers.append(parse_answer(text))
print("Parsed:", new_answers)
print("Vote:", vote_answers(new_answers))

new_records = []
for text in new_fixtures:
    new_records.append(score_candidate(text, new_expected))
for r in new_records:
    print(r["answer"], r["score"], r["status"])

# Remove all passing fixtures
failing_only = [r for r in new_records if r["score"] == 0]
print("Selection from failures only:", select_best(failing_only))

New reference count: 20
Parsed: [20, 18, 20]
Vote: {'answer': 20, 'counts': {20: 2, 18: 1}, 'status': 'selected'}
20 1 correct
18 0 wrong answer
20 1 correct
Selection from failures only: None


### Process Check Practice

In [14]:
# [24, 18, 20]: 24 == 24 True, 24-6=18 True, 18+2=20 True
print("All valid:", check_process([24, 18, 20], 24, 6, 2))

# [24, 19, 20]: 24 == 24 True, 24-6=18 not 19 False, 19+2=21 not 20 False
print("Right final, bad transitions:", check_process([24, 19, 20], 24, 6, 2))

All valid: [True, True, True]
Right final, bad transitions: [True, False, False]


## Summary of Evidence

**What I authored vs. what came from a model:** Every input in this notebook was authored by hand. The five `scripted` fixtures and the three `new_fixtures` are predetermined strings, not model outputs. Steps 10 and 11 have `RUN_LIVE` and `RUN_SAMPLING` set to `False`, so no model requests were made.

**Dictionary lookup:** In `score_candidate`, the expression `record["answer"]` retrieves the value associated with the key `"answer"` from the dictionary `record`. This lets us store multiple related values (text, answer, score, status) in one object and access each by name.

**Loop:** In Step 4, `for text in scripted:` takes one string at a time from the `scripted` list, assigns it to `text`, runs `parse_answer(text)`, and appends the result to `answers`. The loop runs five times because the list has five items.

**Return statement:** In `select_best`, `return record` immediately ends the function and passes that record back to the caller as soon as the first candidate with score 1 is found. If no candidate passes, the loop finishes and the final `return None` runs instead.

### Exit Ticket

- **Training vs. inference:** Training changes the model's weights using data. Inference uses existing weights to generate output.
- **Zero-shot vs. one-shot:** Zero-shot gives the model only the question. One-shot includes one worked example before the question.
- **Voting vs. verification:** Voting picks the most frequent answer (agreement). Verification checks against independent evidence (correctness). Agreement can be wrong when the same mistake repeats.
- **Process vs. outcome checks:** An outcome check sees only the final answer. A process check examines the intermediate steps and transitions.

In a Chapter 2 message-and-response program, a verifier fits after receiving the raw response and before presenting an accepted answer. The raw response is kept for inspection. Repeated candidate requests happen before the selection step.

## Clean Run Verification

After Restart Kernel and Run All with both live switches `False`:
- Parsed values: `[13, 13, 16, 13, None]`
- Vote selects 13 (wrong)
- Verifier selects 16 (correct)
- Process check: `[True, True, True]` and `[True, False, False]`